# Logistic Regression with Python

In this notebook we implement logistic regression for binary classification from scratch.
We introduce the binary cross-entropy (logistic) loss and derive the gradients used in gradient descent.

### Section 1: Binary classification (2D) - separable clusters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
plt.style.use("seaborn-v0_8")

## Generate a separable 2D dataset

We generate two Gaussian clusters in 2D (label 0 and 1) so that we can visualize the decision boundary.

In [ ]:
# Create two clearly separated clusters
X, y = make_blobs(n_samples=300, centers=[(-3, -3), (3, 3)], cluster_std=1.0, random_state=42)
# Plot the dataset
plt.figure(figsize=(6,6))
plt.scatter(X[y==0,0], X[y==0,1], color='blue', alpha=0.6, label='class 0')
plt.scatter(X[y==1,0], X[y==1,1], color='red', alpha=0.6, label='class 1')
plt.legend()
plt.title('Separable 2D dataset (binary classification)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.grid(True, alpha=0.3)
plt.show()

## Logistic model and binary cross-entropy (BCE) loss

Model (sigmoid logistic):

The model outputs a probability

$$
p\bigl(y=1\mid x;\boldsymbol{\beta}\bigr)=\sigma(z),\qquad z=\beta_0+\sum_{j=1}^d\beta_j x_j=\mathbf{x}_{\mathrm{design}}^\top\boldsymbol{\beta}
$$

where $\sigma(z)=\dfrac{1}{1+e^{-z}}$ is the sigmoid function and the design vector is $\mathbf{x}_{\mathrm{design}}=[1,x_1,\dots,x_d]^\top$ (leading 1 for the intercept).

Binary cross-entropy (average over the dataset):

$$
L(\boldsymbol{\beta})=-\frac{1}{N}\sum_{i=1}^N\left[ y_i\log p_i + (1-y_i)\log(1-p_i)\right],\qquad p_i=\sigma\bigl(\mathbf{x}_{i,\mathrm{design}}^\top\boldsymbol{\beta}\bigr)
$$

Vector form (useful for implementation):

$$
\mathbf{p}=\sigma\bigl(X_{\mathrm{design}}\boldsymbol{\beta}\bigr),\qquad L(\boldsymbol{\beta})=-\frac{1}{N}\left(\mathbf{y}^\top\log\mathbf{p} + (\mathbf{1}-\mathbf{y})^\top\log(\mathbf{1}-\mathbf{p})\right).
$$

Gradients (component-wise and vectorized): for each parameter $\beta_j$ (with $x_{i0}=1$ for the intercept)

$$
\frac{\partial L}{\partial \beta_j}=\frac{1}{N}\sum_{i=1}^N (p_i-y_i) x_{ij},
$$

$$
\nabla_{\boldsymbol{\beta}} L=\frac{1}{N}X_{\mathrm{design}}^\top(\mathbf{p}-\mathbf{y}).
$$

#### Notes:
- For numerical stability clip probabilities to $(\varepsilon,1-\varepsilon)$ before evaluating logarithms, or implement numerically stable log-sum-exp style computations.
- Use the vectorized gradient above to implement gradient descent updates: $\boldsymbol{\beta}\leftarrow\boldsymbol{\beta}-\alpha\nabla_{\boldsymbol{\beta}}L$.

Your task: implement the sigmoid, the BCE loss and the gradient (using these formulas) and then implement gradient descent to learn $\boldsymbol{\beta}$.

In [ ]:
# Exercise: Implement the model functions below using the formulas from the markdown cell

def sigmoid(z):
    """
    Numerically stable sigmoid function.

    Parameters
    ----------
    z : array-like
        Input values.

    Returns
    -------
    s : ndarray
        Sigmoid applied elementwise.
    """
    # Student implementation goes here
    raise NotImplementedError


def compute_bce_loss(X_design, y, beta):
    """
    Compute average binary cross-entropy loss.

    Parameters
    ----------
    X_design : ndarray, shape (n_samples, n_features+1)
        Design matrix (including column of ones for intercept).
    y : ndarray, shape (n_samples,)
        Binary labels (0 or 1).
    beta : ndarray, shape (n_features+1,)
        Parameter vector (intercept first).

    Returns
    -------
    loss : float
        Average BCE loss.
    """
    # Student implementation goes here
    raise NotImplementedError


def gradients(X_design, y, beta):
    """
    Compute gradient of BCE loss w.r.t. beta (vectorized):
    """
    # Student implementation goes here
    raise NotImplementedError


def fit_logistic_regression_gd(X, y, learning_rate=0.1, n_steps=1000):
    """
    Fit logistic regression using gradient descent. X should NOT include intercept column.

    Parameters
    ----------
    X : ndarray, shape (n_samples, n_features)
    y : ndarray, shape (n_samples,)
    learning_rate : float
    n_steps : int

    Returns
    -------
    beta : ndarray, shape (n_features+1,)
        Learned parameters (intercept first).
    loss_history : list
        Recorded loss (every k iterations if desired).
    """
    # Student implementation goes here
    raise NotImplementedError


# Example usage scaffold (complete these steps after implementing above functions):
# 1. Split data: X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
# 2. Call: beta, loss_hist = fit_logistic_regression_gd(X_train, y_train, learning_rate=0.1, n_steps=1000)
# 3. Evaluate: create X_design_test = np.hstack((np.ones((X_test.shape[0],1)), X_test))
#    probs = sigmoid(X_design_test.dot(beta)); y_pred = (probs >= 0.5).astype(int)
# 4. Calculate accuracy and plot loss, decision boundary, etc.


Use the function bellow to visualize the decision regions according to your model. What can you say about the boundary between these regions?

In [ ]:
def plot_decision_regions(params, X, y, grid_res=200, cmap='RdBu', proba_contours=[0.5], ax=None, show_prob=True):
    """
    Plot decision regions and probability contours for a logistic regression model (2D inputs).

    This generalized function supports:
    - Binary classification: `params` is a 1D array (d+1,) representing beta (intercept first). In this
      case it shows the probability P(y=1) and optional probability contours.
    - Multiclass classification: `params` is a 2D array (d+1, K) representing the parameter matrix B.
      The function visualizes the predicted class regions (argmax) and shows a discrete colorbar.

    Parameters
    ----------
    params : array-like, shape (d+1,) or (d+1, K)
        Parameter vector (binary) or parameter matrix (multiclass) with intercept first.
    X : ndarray, shape (n_samples, 2)
        Input features (must be 2-dimensional for visualization).
    y : ndarray, shape (n_samples,)
        Labels (integers 0..K-1 for multiclass or 0/1 for binary).
    grid_res : int, default=200
        Resolution of the evaluation grid.
    cmap : str, default='RdBu'
        Colormap for probability field (binary) or base colormap for multiclass.
    proba_contours : list, default=[0.5]
        Probability levels to draw contour lines for (binary case only).
    ax : matplotlib.axes.Axes, optional
        Axes to draw on. If None, a new figure/axes will be created.
    show_prob : bool, default=True
        For binary case: show probability heatmap and contours. For multiclass: shows class regions (argmax).
    """

    from scipy.special import expit as _sigmoid
    from scipy.special import softmax as _softmax

    params = np.asarray(params)
    X = np.asarray(X)
    y = np.asarray(y).ravel()

    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError('plot_decision_regions supports only 2D input features (n_samples, 2)')

    d = X.shape[1]
    # Validate params shape
    if params.ndim == 1:
        if params.shape[0] != d + 1:
            raise ValueError('params must have shape (d+1,) for binary case')
        mode = 'binary'
    elif params.ndim == 2:
        if params.shape[0] != d + 1:
            raise ValueError('params must have shape (d+1, K) for multiclass case')
        mode = 'multiclass'
        K = params.shape[1]
    else:
        raise ValueError('params must be 1D (binary) or 2D (multiclass)')

    x_min, x_max = X[:,0].min() - 1.0, X[:,0].max() + 1.0
    y_min, y_max = X[:,1].min() - 1.0, X[:,1].max() + 1.0

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, grid_res),
                          np.linspace(y_min, y_max, grid_res))
    grid = np.column_stack((xx.ravel(), yy.ravel()))
    X_design_grid = np.hstack(( np.ones((grid.shape[0],1)), grid ))

    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=(7,6))
        created_fig = True

    if mode == 'binary':
        probs = _sigmoid(X_design_grid.dot(params))
        Z = probs.reshape(xx.shape)

        if show_prob:
            cf = ax.contourf(xx, yy, Z, levels=50, cmap=cmap, alpha=0.6)
            if proba_contours is not None and len(proba_contours) > 0:
                ax.contour(xx, yy, Z, levels=proba_contours, colors='k', linewidths=1.5)
            if created_fig:
                cbar = plt.colorbar(cf, ax=ax)
                cbar.set_label('P(y=1)')
        else:
            # Show decision boundary only
            Z_bin = (Z >= 0.5).astype(int)
            cf = ax.contourf(xx, yy, Z_bin, levels=[-0.5, 0.5, 1.5], cmap=cmap, alpha=0.3)

        # scatter the data
        ax.scatter(X[y==0,0], X[y==0,1], color='blue', edgecolor='k', label='class 0', alpha=0.8)
        ax.scatter(X[y==1,0], X[y==1,1], color='red', edgecolor='k', label='class 1', alpha=0.8)

        ax.set_title('Decision regions (P(y=1))')

    else:  # multiclass
        scores = X_design_grid.dot(params)  # (n_grid, K)
        P = _softmax(scores, axis=1)
        y_pred_grid = np.argmax(P, axis=1)
        Z = y_pred_grid.reshape(xx.shape)

        cmap_disc = plt.cm.get_cmap('tab10', K)
        cf = ax.contourf(xx, yy, Z, levels=np.arange(-0.5, K + 0.5, 1), cmap=cmap_disc, alpha=0.5)

        # plot training points colored by true class
        colors = [cmap_disc(i) for i in range(K)]
        for k in range(K):
            ax.scatter(X[y==k,0], X[y==k,1], color=colors[k], edgecolor='k', label=f'class {k}', alpha=0.85)

        ax.set_title('Decision regions (predicted class)')

        if created_fig:
            # discrete colorbar for classes
            cbar = plt.colorbar(cf, ax=ax, ticks=range(K))
            cbar.set_ticks(range(K))
            cbar.set_ticklabels([str(i) for i in range(K)])
            cbar.set_label('Predicted class')

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.legend()

    if created_fig:
        plt.show()


# Example usage (after implementing sigmoid and fitting beta):
# beta, loss_hist = fit_logistic_regression_gd(X_train, y_train, learning_rate=0.1, n_steps=1000)
# plot_decision_regions(beta, X_test, y_test)


## Generate and visualize a 3-class (3-blob) dataset

We create a synthetic 3-class dataset using `make_blobs` to visualize a multiclass classification scenario suitable for the softmax model. The variables `X_multi` and `y_multi` will be available for subsequent exercises.

In [ ]:
# Use clear, separated centers to make the classes visually distinct
X_multi, y_multi = make_blobs(n_samples=450,
                              centers=[(-4, 0), (0, 4), (4, 0)],
                              cluster_std=1.2,
                              random_state=42)

plt.figure(figsize=(6,6))
sc = plt.scatter(X_multi[:,0], X_multi[:,1], c=y_multi, cmap='tab10', s=40, edgecolor='k', alpha=0.85)
plt.legend(*sc.legend_elements(), title='class')
plt.title('3-class dataset (for multiclass / softmax exercises)')
plt.xlabel('x1')
plt.ylabel('x2')
plt.grid(True, alpha=0.3)
plt.show()

# Quick shape overview for students
print('X_multi.shape =', X_multi.shape, 'y_multi.shape =', y_multi.shape)

## Multiclass logistic regression (softmax / multinomial)

For problems with $K>2$ classes we model the conditional distribution using the softmax (multinomial logistic) model.

For input $\mathbf{x}_{i,\mathrm{design}}$ and parameter matrix $B\in\mathbb{R}^{(d+1)\times K}$ (each column $\boldsymbol{\beta}^{(k)}$ corresponds to class $k$) the class probabilities are:

$$
p_{ik}=p(y_i=k\mid \mathbf{x}_i;B)=\mathrm{softmax}_k\bigl(\mathbf{x}_{i,\mathrm{design}}^\top B\bigr)=\frac{\exp\bigl(\mathbf{x}_{i,\mathrm{design}}^\top\boldsymbol{\beta}^{(k)}\bigr)}{\sum_{\ell=1}^K\exp\bigl(\mathbf{x}_{i,\mathrm{design}}^\top\boldsymbol{\beta}^{(\ell)}\bigr)}\,.
$$

The multiclass (categorical) cross-entropy (log loss) averaged over the dataset is:

$$
L(B)=-\frac{1}{N}\sum_{i=1}^N\sum_{k=1}^K y_{ik}\log p_{ik}
$$

where $y_{ik}$ is the one-hot indicator for the true class of example $i$.

The gradient with respect to the parameter vector for class $k$ is:

$$
\nabla_{\boldsymbol{\beta}^{(k)}} L = \frac{1}{N} X_{\mathrm{design}}^\top(\mathbf{p}^{(k)}-\mathbf{y}^{(k)})
$$

where $\mathbf{p}^{(k)}=(p_{1k},\dots,p_{Nk})^\top$ and $\mathbf{y}^{(k)}$ is the one-hot column for class $k$.  In matrix form (all classes):

$$
\nabla_B L = \frac{1}{N} X_{\mathrm{design}}^\top (P - Y)
$$

with $P$ the $N\times K$ matrix of predicted probabilities and $Y$ the $N\times K$ one-hot label matrix.

Hints for implementation:
- Use a [numerically stable softmax](https://en.wikipedia.org/wiki/Softmax_function#Numerical_algorithms) (subtract max per row before exponentiating).
- Represent parameters as a matrix $B$ and compute matrix gradients as shown above.
- Accept labels either as integer class indices or as a one-hot matrix (provide helper to convert).

Your task: implement `softmax`, `compute_multiclass_loss`, `gradients_multiclass`, and `fit_multinomial_gd` following the formulas above.

In [ ]:
def softmax(Z):
    """
    Numerically stable softmax applied row-wise.

    Parameters
    ----------
    Z : ndarray, shape (n_samples, K)
        Unnormalized log-probabilities (scores) for each class.

    Returns
    -------
    P : ndarray, shape (n_samples, K)
        Row-wise softmax probabilities that sum to 1 per row.
    """
    # Student implementation goes here
    raise NotImplementedError


def compute_multiclass_loss(X_design, Y_onehot, B):
    """
    Compute average categorical cross-entropy loss for multiclass predictions.

    Parameters
    ----------
    X_design : ndarray, shape (n_samples, d+1)
        Design matrix with intercept column.
    Y_onehot : ndarray, shape (n_samples, K)
        One-hot encoded labels.
    B : ndarray, shape (d+1, K)
        Parameter matrix (columns are class parameter vectors).

    Returns
    -------
    loss : float
        Average cross-entropy loss.
    """
    # Student implementation goes here
    raise NotImplementedError


def gradients_multiclass(X_design, Y_onehot, B):
    """
    Compute gradient of multiclass cross-entropy w.r.t. B.

    Returns
    -------
    grad : ndarray, shape (d+1, K)
        Gradient matrix of the same shape as B.
    """
    # Student implementation goes here
    raise NotImplementedError


def fit_multinomial_gd(X, y, K, learning_rate=0.1, n_steps=1000):
    """
    Fit multinomial logistic regression using gradient descent.

    Parameters
    ----------
    X : ndarray, shape (n_samples, d)
        Input features (no intercept column).
    y : ndarray, shape (n_samples,)
        Integer class labels in 0..K-1.
    K : int
        Number of classes.
    learning_rate : float
    n_steps : int

    Returns
    -------
    B : ndarray, shape (d+1, K)
        Learned parameter matrix (intercept first).
    loss_history : list
        Recorded loss values.
    """
    # Student implementation goes here
    raise NotImplementedError


# Example scaffold:
# Y_onehot = to_one_hot(y_train, K)
# B, loss_hist = fit_multinomial_gd(X_train, y_train, K=3, learning_rate=0.1, n_steps=1000)
# P = softmax(X_design.dot(B)); y_pred = np.argmax(P, axis=1)


### Example: Fit scikit-learn multinomial logistic and visualize decision regions

This example fits a multinomial logistic regression from scikit-learn to the 3-blob dataset (`X_multi`, `y_multi`). It converts the fitted parameters into the notebook's parameter-matrix format `B` (intercept first, shape `(d+1, K)`) and calls `plot_decision_regions` to visualize the predicted class regions.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Fit multinomial logistic regression (multiclass)
clf = LogisticRegression(solver='lbfgs', C=1.0, max_iter=200)
clf.fit(X_multi, y_multi)

# Convert sklearn parameters to B with intercept first (shape: d+1, K)
B = np.vstack((clf.intercept_[None, :], clf.coef_.T))
print('B.shape =', B.shape)

# Quick evaluation on the training set
train_acc = clf.score(X_multi, y_multi)
print(f'Training accuracy on 3-blob dataset: {train_acc:.3f}')

# Visualize decision regions using the generalized plotting helper
plot_decision_regions(B, X_multi, y_multi)

## Non-linear example: concentric circles (requires non-linear boundary)

This example uses a concentric-circle dataset (binary) to show that a linear logistic model cannot separate the classes, while a non-linear model (RBF SVM) can.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.svm import SVC

# Create non-linearly separable dataset (concentric circles)
X_nl, y_nl = make_circles(n_samples=300, factor=0.5, noise=0.08, random_state=0)

plt.figure(figsize=(6,6))
plt.scatter(X_nl[y_nl==0,0], X_nl[y_nl==0,1], color='blue', label='class 0', edgecolor='k', alpha=0.8)
plt.scatter(X_nl[y_nl==1,0], X_nl[y_nl==1,1], color='red', label='class 1', edgecolor='k', alpha=0.8)
plt.title('Concentric circles (not linearly separable)')
plt.xlabel('x1'); plt.ylabel('x2')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

# Fit linear logistic regression
clf_lin = LogisticRegression()
clf_lin.fit(X_nl, y_nl)
beta_lin = np.hstack((clf_lin.intercept_, clf_lin.coef_.ravel()))
print('Logistic regression training accuracy:', clf_lin.score(X_nl, y_nl))

# Visualize linear decision boundary (will be linear and fail to separate concentric classes)
plot_decision_regions(beta_lin, X_nl, y_nl)

# For contrast: fit an RBF SVM to show non-linear boundary
clf_svc = SVC(kernel='rbf', probability=True)
clf_svc.fit(X_nl, y_nl)
print('RBF SVM training accuracy:', clf_svc.score(X_nl, y_nl))

# Decision surface for SVM (probability for class 1)
x_min, x_max = X_nl[:,0].min()-1, X_nl[:,0].max()+1
y_min, y_max = X_nl[:,1].min()-1, X_nl[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.column_stack((xx.ravel(), yy.ravel()))
Z = clf_svc.predict_proba(grid)[:,1].reshape(xx.shape)

plt.figure(figsize=(7,6))
cf = plt.contourf(xx, yy, Z, levels=50, cmap='RdBu', alpha=0.6)
plt.scatter(X_nl[y_nl==0,0], X_nl[y_nl==0,1], color='blue', edgecolor='k', label='class 0', alpha=0.8)
plt.scatter(X_nl[y_nl==1,0], X_nl[y_nl==1,1], color='red', edgecolor='k', label='class 1', alpha=0.8)
plt.title('RBF SVM decision function (probability for class 1)')
cbar = plt.colorbar(cf); cbar.set_label('P(class 1)')
plt.legend(); plt.show()

### Exercise: try transforming the inputs

Consider applying a feature transform before fitting logistic regression. A suitable transform will map the original 2D inputs into new features that capture the geometric structure of the classes (for example, features that depend on radius or low-degree polynomials). Implement a transform function, re-fit logistic regression on the transformed features, and visualize the probability contour in the original 2D plane.

In [ ]:
# Scaffold (student implementation required)

def feature_transform(X):
    """Student-implemented transform.

    Parameters
    ----------
    X : ndarray, shape (n_samples, 2)
        Original 2D inputs.

    Returns
    -------
    X_trans : ndarray, shape (n_samples, d')
        Transformed features suitable for linear separation.

    Notes
    -----
    - Implement and return transformed features here.
    """
    # Student implementation goes here
    raise NotImplementedError
